# 01 — Data Preprocessing & Merge

Builds a reproducible tabular merge of social profiles and transactions.

**Decisions**
- Normalize `customer_id` (strip + lowercase) **before** any merge.
- Social categoricals: strip + lowercase; missing text → `unknown`.
- Transactions: coerce amount/timestamp; drop non-positive amounts and bad timestamps.
- Soft outlier winsorize on `amount` at 1st/99th percentile.
- RFM features: `total_spent`, `average_spent`, `purchase_count`, `days_since_last_purchase`, plus `fraud_rate`.
- `product_category` = mode of TX `category` (kept in export; notebook 02 must not train with same-source spend features).
- Recency uses a fixed reference date (`2025-04-01`) for reproducibility.
- `merged_dataset.csv` is **intentionally platform-level** (one row per social-platform row).
- `modeling_dataset.csv` is the **customer-level** training frame (one row per customer, no ID duplication).


In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
SOCIAL_PATH = DATA_DIR / "customer_social_profiles.csv"
TX_PATH = DATA_DIR / "customer_transactions.csv"
OUT_PATH = DATA_DIR / "merged_dataset.csv"
MODELING_PATH = DATA_DIR / "modeling_dataset.csv"

REQUIRED_SOCIAL = [
    "customer_id",
    "social_media_platform",
    "engagement_score",
    "purchase_interest_score",
    "review_sentiment",
]
REQUIRED_TX = [
    "transaction_id",
    "customer_id",
    "timestamp",
    "amount",
    "category",
    "is_fraud",
]


## 1. Load with validation


In [2]:
def load_csv(path: Path, required_columns: list[str], name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} not found at {path.resolve()}")
    try:
        df = pd.read_csv(path, encoding="utf-8", sep=",")
    except Exception as exc:
        raise RuntimeError(f"Failed to read {name} from {path}") from exc

    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing required columns: {missing}")
    if df.empty:
        raise ValueError(f"{name} is empty")

    print(f"[OK] Loaded {name}")
    print(f"  path: {path.resolve()}")
    print(f"  shape: {df.shape}")
    print(f"  dtypes:\n{df.dtypes.to_string()}")
    return df


def report_data_quality(df: pd.DataFrame, name: str, id_col: str = "customer_id") -> None:
    """Pre-preprocessing validation summary (does not mutate)."""
    n_dup_rows = int(df.duplicated().sum())
    n_dup_ids = int(df[id_col].duplicated().sum()) if id_col in df.columns else -1
    print(f"\n=== Data quality: {name} ===")
    print(f"  dimensions: {df.shape[0]} rows x {df.shape[1]} cols")
    print(f"  missing values:\n{df.isnull().sum().to_string()}")
    print(f"  duplicate rows: {n_dup_rows}")
    if id_col in df.columns:
        print(f"  duplicate {id_col} values: {n_dup_ids}")
        print(f"  unique {id_col}: {df[id_col].nunique()}")


social_df = load_csv(SOCIAL_PATH, REQUIRED_SOCIAL, "Social profiles")
transactions_df = load_csv(TX_PATH, REQUIRED_TX, "Transactions")

report_data_quality(social_df, "Social profiles")
report_data_quality(transactions_df, "Transactions")
print("\nSocial head:")
print(social_df.head())
print("\nTransactions head:")
print(transactions_df.head())


[OK] Loaded Social profiles
  path: D:\school\ML\Pipeline\Formative 2\multimodal-authentication\data\customer_social_profiles.csv
  shape: (155, 5)
  dtypes:
customer_id                 object
social_media_platform       object
engagement_score             int64
purchase_interest_score    float64
review_sentiment            object
[OK] Loaded Transactions
  path: D:\school\ML\Pipeline\Formative 2\multimodal-authentication\data\customer_transactions.csv
  shape: (328, 12)
  dtypes:
transaction_id     object
customer_id        object
timestamp          object
amount            float64
currency           object
merchant           object
category           object
channel            object
device_type        object
location           object
status             object
is_fraud             bool

=== Data quality: Social profiles ===
  dimensions: 155 rows x 5 cols
  missing values:
customer_id                0
social_media_platform      0
engagement_score           0
purchase_interest_score   

## 2. Clean & standardize


In [3]:
# --- IDs first (must happen before any merge) ---
social_df["customer_id"] = social_df["customer_id"].astype(str).str.strip().str.lower()
transactions_df["customer_id"] = (
    transactions_df["customer_id"].astype(str).str.strip().str.lower()
)

# --- Social categoricals ---
for col in ["social_media_platform", "review_sentiment"]:
    social_df[col] = social_df[col].astype(str).str.strip().str.lower()

social_df["engagement_score"] = pd.to_numeric(
    social_df["engagement_score"], errors="coerce"
)
social_df["purchase_interest_score"] = pd.to_numeric(
    social_df["purchase_interest_score"], errors="coerce"
)

n_social_before = len(social_df)
social_df = social_df.drop_duplicates()
print(f"Social duplicates removed: {n_social_before - len(social_df)}")

social_df["social_media_platform"] = social_df["social_media_platform"].fillna("unknown")
social_df["review_sentiment"] = social_df["review_sentiment"].fillna("unknown")
social_df["engagement_score"] = social_df["engagement_score"].fillna(
    social_df["engagement_score"].median()
)
social_df["purchase_interest_score"] = social_df["purchase_interest_score"].fillna(
    social_df["purchase_interest_score"].median()
)

assert social_df["customer_id"].notna().all(), "Null customer_id in social_df"
assert social_df.isnull().sum().sum() == 0, social_df.isnull().sum()

# --- Transactions ---
n_tx_before = len(transactions_df)
transactions_df = transactions_df.drop_duplicates()
print(f"TX duplicates removed: {n_tx_before - len(transactions_df)}")

transactions_df["amount"] = pd.to_numeric(transactions_df["amount"], errors="coerce")
transactions_df["timestamp"] = pd.to_datetime(
    transactions_df["timestamp"], errors="coerce"
)
transactions_df["category"] = (
    transactions_df["category"].astype(str).str.strip().str.lower()
)

bad_amount = transactions_df["amount"].isna() | (transactions_df["amount"] <= 0)
bad_ts = transactions_df["timestamp"].isna()
print(f"Dropping TX with bad amount: {int(bad_amount.sum())}")
print(f"Dropping TX with bad timestamp: {int(bad_ts.sum())}")
transactions_df = transactions_df.loc[~bad_amount & ~bad_ts].copy()

lo, hi = transactions_df["amount"].quantile([0.01, 0.99])
n_out = int(((transactions_df["amount"] < lo) | (transactions_df["amount"] > hi)).sum())
transactions_df["amount"] = transactions_df["amount"].clip(lo, hi)
print(f"Amounts winsorized (1-99%): {n_out} values clipped to [{lo:.2f}, {hi:.2f}]")

assert transactions_df["customer_id"].notna().all()
assert (transactions_df["amount"] > 0).all()
assert transactions_df["timestamp"].notna().all()
print("Clean social:", social_df.shape, "| Clean TX:", transactions_df.shape)


Social duplicates removed: 5
TX duplicates removed: 0
Dropping TX with bad amount: 0
Dropping TX with bad timestamp: 0
Amounts winsorized (1-99%): 8 values clipped to [158.82, 24007.23]
Clean social: (150, 5) | Clean TX: (328, 12)


## 3. Feature engineering (RFM + fraud + preferred category)


In [4]:
transaction_features = (
    transactions_df.groupby("customer_id", as_index=False)
    .agg(
        total_spent=("amount", "sum"),
        average_spent=("amount", "mean"),
        purchase_count=("amount", "count"),
        last_purchase=("timestamp", "max"),
        fraud_rate=("is_fraud", "mean"),
    )
)

# Fixed reference date so recency is reproducible across runs
ref_day = pd.Timestamp("2025-04-01")
transaction_features["days_since_last_purchase"] = (
    ref_day - transaction_features["last_purchase"]
).dt.days

product_category = (
    transactions_df.groupby("customer_id")["category"]
    .agg(lambda s: s.value_counts().index[0])
    .reset_index(name="product_category")
)
transaction_features = transaction_features.merge(
    product_category, on="customer_id", how="left"
)

required_tx_feats = [
    "total_spent",
    "average_spent",
    "purchase_count",
    "days_since_last_purchase",
    "fraud_rate",
    "product_category",
]
missing_feats = [c for c in required_tx_feats if c not in transaction_features.columns]
assert not missing_feats, missing_feats
assert transaction_features[required_tx_feats].isnull().sum().sum() == 0, (
    transaction_features[required_tx_feats].isnull().sum()
)

print("Transaction feature rows:", len(transaction_features))
print("Target class distribution (customer-level):")
print(transaction_features["product_category"].value_counts())
print(transaction_features.head())


Transaction feature rows: 84
Target class distribution (customer-level):
product_category
other        15
finance      14
retail        9
transport     9
clothing      7
ecommerce     6
fuel          5
health        5
food          5
groceries     4
wholesale     3
travel        2
Name: count, dtype: int64
  customer_id  total_spent  average_spent  purchase_count       last_purchase  \
0        a100     34000.36   17000.180000               2 2025-03-04 04:05:00   
1        a101      6900.05    2300.016667               3 2025-03-21 01:51:00   
2        a102     25993.46    6498.365000               4 2025-03-09 08:58:00   
3        a103     23695.70    5923.925000               4 2025-03-22 06:14:00   
4        a104      3134.38    1567.190000               2 2025-03-02 15:28:00   

   fraud_rate  days_since_last_purchase product_category  
0        0.00                        27            other  
1        0.00                        10        ecommerce  
2        0.25               

## 4. Social aggregates + merge

`merged_df` stays **platform-level** (one social-profile row per platform × customer) for authentication / inspection.
A separate **customer-level** `modeling_df` is built next so recommendation training never duplicates IDs.


In [5]:
engagement = (
    social_df.groupby("customer_id", as_index=False)
    .agg(average_engagement=("engagement_score", "mean"))
)
platform_count = (
    social_df.groupby("customer_id")["social_media_platform"]
    .nunique()
    .reset_index(name="platform_count")
)
main_platform = (
    social_df.groupby("customer_id")["social_media_platform"]
    .agg(lambda s: s.value_counts().index[0])
    .reset_index(name="main_platform")
)

n_before = len(social_df)
merged_df = social_df.merge(transaction_features, on="customer_id", how="left")
merged_df = merged_df.merge(engagement, on="customer_id", how="left")
merged_df = merged_df.merge(platform_count, on="customer_id", how="left")
merged_df = merged_df.merge(main_platform, on="customer_id", how="left")
merged_df = merged_df.drop_duplicates()

assert len(merged_df) == n_before, "Merge changed social row count unexpectedly"

tx_cols = [
    "total_spent",
    "average_spent",
    "purchase_count",
    "days_since_last_purchase",
    "fraud_rate",
]
match_rate = merged_df["total_spent"].notna().mean()
print(f"TX match rate (non-null total_spent): {match_rate:.1%}")
assert match_rate >= 0.5, (
    f"Suspiciously low TX match rate ({match_rate:.1%}) — check customer_id casing"
)

# Customers with no TX: neutral defaults
merged_df["total_spent"] = merged_df["total_spent"].fillna(0.0)
merged_df["average_spent"] = merged_df["average_spent"].fillna(0.0)
merged_df["purchase_count"] = merged_df["purchase_count"].fillna(0).astype(int)
merged_df["days_since_last_purchase"] = merged_df["days_since_last_purchase"].fillna(-1)
merged_df["fraud_rate"] = merged_df["fraud_rate"].fillna(0.0)
merged_df["product_category"] = merged_df["product_category"].fillna("unknown")

assert merged_df[tx_cols].isnull().sum().sum() == 0, merged_df[tx_cols].isnull().sum()
assert (merged_df["customer_id"] == merged_df["customer_id"].str.lower()).all()

print("Platform-level merged shape:", merged_df.shape)
print("Unique customers in merged:", merged_df["customer_id"].nunique())
print("Nulls:\n", merged_df.isnull().sum())
print(merged_df.head())


TX match rate (non-null total_spent): 100.0%
Platform-level merged shape: (150, 15)
Unique customers in merged: 84
Nulls:
 customer_id                 0
social_media_platform       0
engagement_score            0
purchase_interest_score     0
review_sentiment            0
total_spent                 0
average_spent               0
purchase_count              0
last_purchase               0
fraud_rate                  0
days_since_last_purchase    0
product_category            0
average_engagement          0
platform_count              0
main_platform               0
dtype: int64
  customer_id social_media_platform  engagement_score  \
0        a178              linkedin                74   
1        a190               twitter                82   
2        a150              facebook                96   
3        a162               twitter                89   
4        a197               twitter                92   

   purchase_interest_score review_sentiment  total_spent  average_spent

## 5. Customer-level modeling dataset

Aggregates social signals to **one row per `customer_id`** for leakage-safe train/test splits.
RFM / `fraud_rate` / `product_category` are already customer-level from Step 3.


In [6]:
modeling_df = (
    social_df.groupby("customer_id", as_index=False)
    .agg(
        engagement_score=("engagement_score", "mean"),
        purchase_interest_score=("purchase_interest_score", "mean"),
        review_sentiment=("review_sentiment", lambda s: s.value_counts().index[0]),
    )
)
modeling_df = modeling_df.merge(engagement, on="customer_id", how="left")
modeling_df = modeling_df.merge(platform_count, on="customer_id", how="left")
modeling_df = modeling_df.merge(main_platform, on="customer_id", how="left")
modeling_df = modeling_df.merge(transaction_features, on="customer_id", how="left")

modeling_df["total_spent"] = modeling_df["total_spent"].fillna(0.0)
modeling_df["average_spent"] = modeling_df["average_spent"].fillna(0.0)
modeling_df["purchase_count"] = modeling_df["purchase_count"].fillna(0).astype(int)
modeling_df["days_since_last_purchase"] = modeling_df["days_since_last_purchase"].fillna(-1)
modeling_df["fraud_rate"] = modeling_df["fraud_rate"].fillna(0.0)
modeling_df["product_category"] = modeling_df["product_category"].fillna("unknown")

assert modeling_df["customer_id"].is_unique, "modeling_df must be one row per customer"
assert modeling_df["product_category"].notna().all()
print("Customer-level modeling shape:", modeling_df.shape)
print("Class distribution:")
print(modeling_df["product_category"].value_counts())
print(modeling_df.head())


Customer-level modeling shape: (84, 14)
Class distribution:
product_category
other        15
finance      14
retail        9
transport     9
clothing      7
ecommerce     6
fuel          5
health        5
food          5
groceries     4
wholesale     3
travel        2
Name: count, dtype: int64
  customer_id  engagement_score  purchase_interest_score review_sentiment  \
0        a100         77.000000                 4.400000         negative   
1        a101         68.000000                 1.000000          neutral   
2        a102         51.000000                 4.800000          neutral   
3        a103         64.333333                 2.866667         positive   
4        a104         83.000000                 2.933333         negative   

   average_engagement  platform_count main_platform  total_spent  \
0           77.000000               2       twitter     34000.36   
1           68.000000               1       twitter      6900.05   
2           51.000000               1 

## 6. Export


In [7]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUT_PATH, index=False, encoding="utf-8")
modeling_df.to_csv(MODELING_PATH, index=False, encoding="utf-8")

reloaded = pd.read_csv(OUT_PATH, encoding="utf-8", sep=",")
assert list(reloaded.columns) == list(merged_df.columns)
assert reloaded["total_spent"].notna().all()
assert (reloaded["total_spent"] >= 0).all()

reloaded_model = pd.read_csv(MODELING_PATH, encoding="utf-8", sep=",")
assert reloaded_model["customer_id"].is_unique
assert len(reloaded_model) == merged_df["customer_id"].nunique()

print(f"Saved platform-level -> {OUT_PATH.resolve()} ({len(reloaded)} rows)")
print(f"Saved modeling      -> {MODELING_PATH.resolve()} ({len(reloaded_model)} rows)")


Saved platform-level -> D:\school\ML\Pipeline\Formative 2\multimodal-authentication\data\merged_dataset.csv (150 rows)


Saved modeling      -> D:\school\ML\Pipeline\Formative 2\multimodal-authentication\data\modeling_dataset.csv (84 rows)
